# Fake News Detection

This notebook is the primary workflow for loading the local dataset, training and evaluating a TF-IDF/logistic-regression baseline, and saving the complete model artifact for later upload to Hugging Face.

The dataset is the [Fake and Real News Dataset on Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?resource=download). Put the downloaded `Fake.csv` and `True.csv` files in the project's `data/` directory. This experiment does not download the data automatically.

> This is a dataset-label classifier, not a fact-checker. Its predictions and confidence scores are not evidence that an article is true or false.

## Configure the run

The notebook uses the project's shared training functions so its preprocessing and saved model remain compatible with the command-line tools.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "fake_news_detector.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "fake_news_detector.py").is_file():
    raise FileNotFoundError("Open this notebook from the project workspace so fake_news_detector.py can be found.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fake_news_detector import load_dataset, load_model, predict_text, save_model, train_model

DATA_DIR = PROJECT_ROOT / "data"
MODEL_PATH = PROJECT_ROOT / "models" / "fake_news_model.joblib"
TEST_SIZE = 0.2
RANDOM_STATE = 42

print(f"Dataset directory: {DATA_DIR}")
print(f"Model output: {MODEL_PATH}")

Dataset directory: C:\ALI\Projects\Project2\data
Model output: C:\ALI\Projects\Project2\models\fake_news_model.joblib


## Load and inspect the data

`Fake.csv` and `True.csv` provide the labels by filename. The default features combine each row's title and article text; date and subject metadata are excluded.

In [2]:
from collections import Counter

texts, labels, text_columns = load_dataset(DATA_DIR)
class_counts = Counter(labels)

print(f"Usable articles: {len(texts):,}")
print(f"Text columns: {text_columns}")
print(f"Class counts: {dict(class_counts)}")

Usable articles: 44,898
Text columns: ['title', 'text']
Class counts: {'fake': 23481, 'real': 21417}


## Train and evaluate

A stratified holdout split is used so each class is represented in both partitions. The TF-IDF vectorizer is fitted only on the training partition to prevent test-set vocabulary leakage.

In [3]:
pipeline, metrics = train_model(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(json.dumps(metrics, indent=2, sort_keys=True))

{
  "accuracy": 0.9918708240534521,
  "classification_report": {
    "accuracy": 0.9918708240534521,
    "fake": {
      "f1-score": 0.9922149941345846,
      "precision": 0.9938047425763725,
      "recall": 0.9906303236797275,
      "support": 4696.0
    },
    "macro avg": {
      "f1-score": 0.9918549047336094,
      "precision": 0.9917849021093075,
      "recall": 0.9919304746316471,
      "support": 8980.0
    },
    "real": {
      "f1-score": 0.9914948153326343,
      "precision": 0.9897650616422424,
      "recall": 0.9932306255835668,
      "support": 4284.0
    },
    "weighted avg": {
      "f1-score": 0.9918714255390886,
      "precision": 0.9918775718501127,
      "recall": 0.9918708240534521,
      "support": 8980.0
    }
  },
  "confusion_matrix": [
    [
      4652,
      44
    ],
    [
      29,
      4255
    ]
  ],
  "f1_macro": 0.9918549047336094,
  "precision_macro": 0.9917849021093075,
  "recall_macro": 0.9919304746316471,
  "test_examples": 8980,
  "train_example

## Save the model artifact

The artifact contains the fitted preprocessing/classification pipeline, input column names, and holdout metrics. It is saved locally at `models/fake_news_model.joblib`; upload that file to Hugging Face when ready. Only load artifacts you created or trust, since `joblib` uses Python serialization.

In [4]:
save_model(
    MODEL_PATH,
    pipeline,
    text_columns=text_columns,
    label_column=None,
    metrics=metrics,
)

print(f"Saved model: {MODEL_PATH}")
print(f"Artifact size: {MODEL_PATH.stat().st_size / (1024 * 1024):.1f} MiB")

Saved model: C:\ALI\Projects\Project2\models\fake_news_model.joblib
Artifact size: 4.7 MiB


## Try a prediction

Replace the example with an article headline and body. The returned confidence is the model's estimated class probability, not a calibrated measure of truthfulness.

In [5]:
artifact = load_model(MODEL_PATH)
example_article = "Officials confirmed the results after reviewing the final report."
prediction = predict_text(artifact["pipeline"], example_article)
prediction

{'label': 'fake',
 'confidence': 0.7835034186251995,
 'probabilities': {'fake': 0.7835034186251995, 'real': 0.21649658137480057}}